# Import Libraries and Define Project Folders

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from keplergl import KeplerGl
import json

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 120

/opt/anaconda3/envs/citibike-dashboard/lib/python3.11/site-packages/keplergl/keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


# Load Data

In [2]:
df = pd.read_csv(
    DATA_DIR / "cbsd_main_2022_02.csv",
    dtype={"start_station_id": "string", "end_station_id": "string"},
    low_memory=False
)

# parse dates safely (errors='coerce' prevents crashes if any weird rows exist)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
df["ended_at"] = pd.to_datetime(df["ended_at"], errors="coerce")

df.shape

(29838166, 15)

In [3]:
daily = pd.read_csv(DATA_DIR / "cbsd_daily_2022_01.csv", parse_dates=["date"])
daily = daily.sort_values("date").set_index("date")

daily.shape

(365, 2)

In [4]:
df["value"] = 1

df_group = (
    df.groupby(["start_station_name", "end_station_name"])["value"]
      .count()
      .reset_index()
      .rename(columns={"value": "trips"})
)

coords = (
    df.groupby(["start_station_name", "end_station_name"])
      .agg(
          start_lat=("start_lat", "first"),
          start_lng=("start_lng", "first"),
          end_lat=("end_lat", "first"),
          end_lng=("end_lng", "first"),
      )
      .reset_index()
)

df_final = df_group.merge(coords, on=["start_station_name", "end_station_name"], how="left")

df_final.head()

,start_station_name,end_station_name,trips,start_lat,start_lng,end_lat,end_lng
0,1 Ave & E 110 St,1 Ave & E 110 St,791,40.792327,-73.9383,40.792327,-73.938300
1,1 Ave & E 110 St,1 Ave & E 18 St,2,40.792327,-73.9383,40.733812,-73.980544
2,1 Ave & E 110 St,1 Ave & E 30 St,4,40.792327,-73.9383,40.741444,-73.975361
3,1 Ave & E 110 St,1 Ave & E 39 St,1,40.792327,-73.9383,40.747140,-73.971130
4,1 Ave & E 110 St,1 Ave & E 44 St,12,40.792327,-73.9383,40.750020,-73.969053


In [5]:
df_final[["start_lat","start_lng","end_lat","end_lng"]].isna().mean()

start_lat    0.0
start_lng    0.0
end_lat      0.0
end_lng      0.0
dtype: float64

In [6]:
from keplergl import KeplerGl
m = KeplerGl(height=700, data={"citi_bike_trips": df_final})
m

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(data={'citi_bike_trips':             start_station_name       end_station_name  trips  start_lat  \
0…

In [9]:
config = m.config
with open(OUTPUTS_DIR / "cbsd_kepler_config.json", "w") as f:
    json.dump(config, f)

m.save_to_html(
    file_name=str(OUTPUTS_DIR / "cbsd_kepler_trips_configured.html"),
    read_only=False,
    config=config
)

Map saved to ../outputs/cbsd_kepler_trips_configured.html!
